In [ ]:
!pip install -q \
transformers \
datasets \
accelerate \
trl \
peft \
bitsandbytes \
qwen-vl-utils

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0" transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 146.9 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
from PIL import Image
import torchao

from datasets import Dataset

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print(torchao.__version__)

0.17.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

PROCESSED_DIR = Path(config["processed_dir"])

train_subset = pd.read_parquet(PROCESSED_DIR / "matched_train.parquet")
val_subset = pd.read_parquet(PROCESSED_DIR / "matched_val.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "matched_test.parquet")

print(len(train_subset))
print(len(val_subset))
print(len(test_df))

print("\nTrain task distribution:")
print(train_subset["task"].value_counts())


91000
9000
5600

Train task distribution:
task
Action Recognition              13000
Instrument Recognition          13000
Phase Recognition               13000
Safety Assessment               13000
Surgical Image Captioning       13000
Tissue and Organ Recognition    13000
Triplet Recognition             13000
Name: count, dtype: int64


In [ ]:
import time, shutil, subprocess
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

DRIVE_CHOLECT50_ARCHIVE = Path("/content/drive/MyDrive/Surgical-VLM/data/CholecT50_raw.zip")  # adjust extension if .tar.gz
LOCAL_ARCHIVE_COPY = Path("/content/CholecT50_raw" + DRIVE_CHOLECT50_ARCHIVE.suffix)
LOCAL_CHOLECT50_DIR = Path("/content/CholecT50")

assert DRIVE_CHOLECT50_ARCHIVE.exists(), (
    f"Expected archive not found at {DRIVE_CHOLECT50_ARCHIVE}. "
    "Upload it to Drive first (see markdown above) before running this cell."
)


print("Copying archive to local disk..")
t0 = time.time()
shutil.copy2(DRIVE_CHOLECT50_ARCHIVE, LOCAL_ARCHIVE_COPY)
print(f"Copy done in {time.time()-t0:.1f}s")

LOCAL_CHOLECT50_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting locally...")
t0 = time.time()
if LOCAL_ARCHIVE_COPY.suffix == ".zip":
    subprocess.run(["unzip", "-q", str(LOCAL_ARCHIVE_COPY), "-d", str(LOCAL_CHOLECT50_DIR)], check=True)
else:
    subprocess.run(["tar", "-xzf", str(LOCAL_ARCHIVE_COPY), "-C", str(LOCAL_CHOLECT50_DIR)], check=True)
print(f"Extract done in {time.time()-t0:.1f}s")

n_files = sum(1 for _ in LOCAL_CHOLECT50_DIR.rglob("*") if _.is_file())
print(f"\n{n_files:,} files extracted to {LOCAL_CHOLECT50_DIR}")

LOCAL_ARCHIVE_COPY.unlink()



Copying archive to local disk..
Copy done in 1122.7s
Extracting locally...
Extract done in 626.7s

100,918 files extracted to /content/CholecT50


In [ ]:
train_dataset = Dataset.from_pandas(
    train_subset[
        ["image_full_path","question","answer"]
    ]
)

val_dataset = Dataset.from_pandas(
    val_subset[
        ["image_full_path","question","answer"]
    ]
)

test_dataset = Dataset.from_pandas(
    test_df[
        ["image_full_path","question","answer"]
    ]
)

In [ ]:
print(val_dataset.shape)

(9000, 3)


In [ ]:
from transformers import BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels = 1024 * 28 * 28
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

for name, module in model.named_modules():
    if any(t in name for t in ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]):
        print(name)

target_modules = [
    name for name, _ in model.named_modules()
    if any(t in name for t in ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    and "visual" not in name
]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=target_modules
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

model.visual.blocks.0.mlp.gate_proj
model.visual.blocks.0.mlp.up_proj
model.visual.blocks.0.mlp.down_proj
model.visual.blocks.1.mlp.gate_proj
model.visual.blocks.1.mlp.up_proj
model.visual.blocks.1.mlp.down_proj
model.visual.blocks.2.mlp.gate_proj
model.visual.blocks.2.mlp.up_proj
model.visual.blocks.2.mlp.down_proj
model.visual.blocks.3.mlp.gate_proj
model.visual.blocks.3.mlp.up_proj
model.visual.blocks.3.mlp.down_proj
model.visual.blocks.4.mlp.gate_proj
model.visual.blocks.4.mlp.up_proj
model.visual.blocks.4.mlp.down_proj
model.visual.blocks.5.mlp.gate_proj
model.visual.blocks.5.mlp.up_proj
model.visual.blocks.5.mlp.down_proj
model.visual.blocks.6.mlp.gate_proj
model.visual.blocks.6.mlp.up_proj
model.visual.blocks.6.mlp.down_proj
model.visual.blocks.7.mlp.gate_proj
model.visual.blocks.7.mlp.up_proj
model.visual.blocks.7.mlp.down_proj
model.visual.blocks.8.mlp.gate_proj
model.visual.blocks.8.mlp.up_proj
model.visual.blocks.8.mlp.down_proj
model.visual.blocks.9.mlp.gate_proj
model.visu

In [ ]:
from PIL import Image

def create_messages(example):

    image = Image.open(example["image_full_path"]).convert("RGB")

    messages = [

        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": example["question"]
                }
            ]
        },

        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": example["answer"]
                }
            ]
        }

    ]

    return messages

In [ ]:
# Check before writing collate function
print(processor.tokenizer.padding_side)

right


In [ ]:
from PIL import Image
import torch

def collate_fn(examples):
    images = [Image.open(ex["image_full_path"]).convert("RGB") for ex in examples]

    prompts = []
    full_texts = []
    for ex, image in zip(examples, images):
        prompt_messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": ex["question"]},
            ],
        }]
        prompt = processor.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt)
        full_texts.append(prompt + ex["answer"])

    batch = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()

    for i, (prompt, image) in enumerate(zip(prompts, images)):
        prompt_only = processor(
            text=[prompt],
            images=[image],
            return_tensors="pt",
        )
        prompt_len = int(prompt_only["attention_mask"][0].sum())

        if processor.tokenizer.padding_side == "left":
            total_len = int(batch["attention_mask"][i].sum())
            pad_len = batch["input_ids"].shape[1] - total_len
            labels[i, : pad_len + prompt_len] = -100
        else:
            labels[i, :prompt_len] = -100

    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

In [ ]:
sample_batch = collate_fn([train_dataset[0], train_dataset[1]])
for i in range(2):
    unmasked = (sample_batch["labels"][i] != -100).sum().item()
    total = sample_batch["attention_mask"][i].sum().item()
    print(f"Sample {i}: {unmasked} unmasked / {total} real tokens "
          f"({unmasked/total:.1%} — should roughly match your answer length, not ~85-90%)")

Sample 0: 2 unmasked / 553 real tokens (0.4% — should roughly match your answer length, not ~85-90%)
Sample 1: 14 unmasked / 558 real tokens (2.5% — should roughly match your answer length, not ~85-90%)


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    train_dataset,
    batch_size=2,
    collate_fn=collate_fn
)

batch = next(iter(loader))

for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([2, 558])
attention_mask torch.Size([2, 558])
mm_token_type_ids torch.Size([2, 558])
pixel_values torch.Size([4080, 1176])
image_grid_thw torch.Size([2, 3])
labels torch.Size([2, 558])


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/Surgical-VLM/checkpoints",

    num_train_epochs=1,

    learning_rate=2e-4,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    gradient_accumulation_steps=1,

    warmup_ratio=0.03,

    weight_decay=0.01,

    logging_steps=25,

    eval_strategy="steps",

    eval_steps=1000,

    save_strategy="steps",

    save_steps=1000,

    save_total_limit=2,

    load_best_model_at_end=True,

    bf16=True,

    fp16=False,

    remove_unused_columns=False,

    report_to="none",

    gradient_checkpointing=True,

    gradient_checkpointing_kwargs={"use_reentrant": False},

    torch_compile=False,

    dataloader_num_workers = 4,

    dataloader_pin_memory=True,

    optim="adamw_torch_fused"

)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=collate_fn

)

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/Surgical-VLM/checkpoints/checkpoint-4000")

Step,Training Loss,Validation Loss
5000,0.000000,nan
5688,0.000000,nan


TrainOutput(global_step=5688, training_loss=0.0, metrics={'train_runtime': 32006.151, 'train_samples_per_second': 2.843, 'train_steps_per_second': 0.178, 'total_flos': 1.6134350317674824e+18, 'train_loss': 0.0, 'epoch': 1.0})

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Surgical-VLM/checkpoints/best_qwen_vl_final"

trainer.save_model(SAVE_DIR)

processor.save_pretrained(SAVE_DIR)

print("Model saved.")

Model saved.


In [ ]:
from PIL import Image

sample = test_dataset[0]

image = Image.open(sample["image_full_path"]).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": sample["question"]}
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    images=[image],
    return_tensors="pt"
)

inputs = inputs.to(model.device)

input_len = inputs["input_ids"].shape[1]

generated_ids = model.generate(
    **inputs,
    max_new_tokens=64
)

new_tokens = generated_ids[:, input_len:]

answer = processor.batch_decode(
    new_tokens,
    skip_special_tokens=True
)[0]

print("Question:", sample["question"])
print("Ground Truth:", sample["answer"])
print("Prediction:", answer)

NameError: name 'processor' is not defined

In [ ]:
!pip install -q \
sentence-transformers \
bert-score \
rouge-score \
nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import torchao

from tqdm.auto import tqdm
from PIL import Image

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration
)

from peft import PeftModel

print(torchao.__version__)

In [ ]:
eval_df = pd.read_parquet(PROCESSED_DIR / "matched_test.parquet").reset_index(drop=True)

print(eval_df.shape)
print(eval_df["task"].value_counts())


(5600, 18)
task
Action Recognition              800
Instrument Recognition          800
Phase Recognition               800
Safety Assessment               800
Surgical Image Captioning       800
Tissue and Organ Recognition    800
Triplet Recognition             800
Name: count, dtype: int64


In [ ]:
import os
from peft import PeftModel
from transformers import BitsAndBytesConfig

CHECKPOINT = "/content/drive/MyDrive/Surgical-VLM/checkpoints/best_qwen_vl_final"

processor = AutoProcessor.from_pretrained(CHECKPOINT)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype = torch.bfloat16
)

model = PeftModel.from_pretrained(base_model, CHECKPOINT)
model.eval()

print("Model loaded.")

_sanity_row = eval_df.iloc[0]
_sanity_image = Image.open(_sanity_row.image_full_path).convert("RGB")
_sanity_messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": _sanity_image},
        {"type": "text", "text": _sanity_row.question}
    ]
}]
_sanity_text = processor.apply_chat_template(
    _sanity_messages, tokenize=False, add_generation_prompt=True
)
_sanity_inputs = processor(
    text=[_sanity_text], images=[_sanity_image], return_tensors="pt"
).to(model.device)

# Generate base model output with adapter disabled
with model.disable_adapter():
    _base_out = model.generate(**_sanity_inputs, max_new_tokens=32)

# Generate adapted model output with adapter enabled
_adapted_out = model.generate(**_sanity_inputs, max_new_tokens=32)

_base_text = processor.batch_decode(_base_out, skip_special_tokens=True)[0]
_adapted_text = processor.batch_decode(_adapted_out, skip_special_tokens=True)[0]

print("Base model output   :", _base_text)
print("Adapted model output:", _adapted_text)
assert _base_text != _adapted_text, (
    "Adapter appears to have no effect, base and adapted outputs are "
    "identical. Check that adapter checkpoint loaded correctly."
)
print("\n Adapter is having an effect (outputs differ from base model).")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model loaded.
Base model output   : system
You are a helpful assistant.
user
Identify the grasper's action in this surgery image.
assistant
The image shows an endoscopic view of a surgical procedure, likely during a procedure involving the gastrointestinal tract. The grasper is being used to manipulate or stabilize tissue
Adapted model output: system
You are a helpful assistant.
user
Identify the grasper's action in this surgery image.
assistant
The grasper is performing a retract action in this surgery image.

 Adapter is having an effect (outputs differ from base model).


In [ ]:
processor.tokenizer.padding_side = "left"

def generate_batch(rows, batch_size=8):
    predictions = []
    for i in tqdm(range(0, len(rows), batch_size)):
        batch_rows = rows.iloc[i:i+batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_rows.image_full_path]
        texts = []
        for img, q in zip(images, batch_rows.question):
            messages = [{"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": q},
            ]}]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

        new_tokens = generated_ids[:, input_len:]
        batch_preds = processor.batch_decode(new_tokens, skip_special_tokens=True)
        predictions.extend(batch_preds)

        # checkpoint partial results every batch — protects a 3h+ eval from a disconnect too
        pd.Series(predictions).to_csv("/content/drive/MyDrive/Surgical-VLM/results/partial_predictions.csv", index=False)

    return predictions

In [ ]:
sample = eval_df.iloc[0]

prediction = generate_answer(

    sample.image_full_path,

    sample.question

)

print()

print("QUESTION")
print(sample.question)

print()

print("GROUND TRUTH")
print(sample.answer)

print()

print("PREDICTION")
print(prediction)


QUESTION
Identify the grasper's action in this surgery image.

GROUND TRUTH
The grasper is performing a retract action in this surgery image.

PREDICTION
The grasper is performing a retract action in this surgery image.


In [ ]:
predictions = generate_batch(eval_df, batch_size=8)

  0%|          | 0/700 [00:00<?, ?it/s]

In [ ]:
results_df = eval_df.copy()

results_df["prediction"] = predictions

results_df = results_df[

    [

        "image_full_path",

        "task",

        "question",

        "answer",

        "prediction"

    ]

]

results_df.rename(

    columns={

        "answer":"ground_truth"

    },

    inplace=True

)

results_df.head()

,image_full_path,task,question,ground_truth,prediction
0,/content/CholecT50/CholecT50/videos/VID110/001...,Action Recognition,Identify the grasper's action in this surgery ...,The grasper is performing a retract action in ...,The grasper is performing a retract action in ...
1,/content/CholecT50/CholecT50/videos/VID68/0015...,Action Recognition,"Given the laparoscopic cholecystectomy image, ...","retract, dissect","retract, dissect, coagulate, aspirate, null ve..."
2,/content/CholecT50/CholecT50/videos/VID36/0002...,Action Recognition,What is the grasper doing in this surgical scene?,The grasper is performing a retract action in ...,The grasper is performing a retract action in ...
3,/content/CholecT50/CholecT50/videos/VID05/0002...,Action Recognition,"Given the laparoscopic cholecystectomy image, ...","retract, dissect","retract, dissect, null verb, null target, null..."
4,/content/CholecT50/CholecT50/videos/VID36/0012...,Action Recognition,"Given the laparoscopic cholecystectomy image, ...",dissect,"dissect, coagulate, retract, dissect, coagulat..."


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Surgical-VLM/results/visionllm_predictions.csv"

os.makedirs(

    os.path.dirname(SAVE_PATH),

    exist_ok=True

)

results_df.to_csv(

    SAVE_PATH,

    index=False

)

print("Saved to:")

print(SAVE_PATH)

Saved to:
/content/drive/MyDrive/Surgical-VLM/results/visionllm_predictions.csv


In [ ]:
results_df = pd.read_csv(

    "/content/drive/MyDrive/Surgical-VLM/results/visionllm_predictions.csv"

)

results_df.head()

,image_full_path,task,question,ground_truth,prediction
0,/content/CholecT50/CholecT50/videos/VID110/001...,Action Recognition,Identify the grasper's action in this surgery ...,The grasper is performing a retract action in ...,The grasper is performing a retract action in ...
1,/content/CholecT50/CholecT50/videos/VID68/0015...,Action Recognition,"Given the laparoscopic cholecystectomy image, ...","retract, dissect","retract, dissect, coagulate, aspirate, null ve..."
2,/content/CholecT50/CholecT50/videos/VID36/0002...,Action Recognition,What is the grasper doing in this surgical scene?,The grasper is performing a retract action in ...,The grasper is performing a retract action in ...
3,/content/CholecT50/CholecT50/videos/VID05/0002...,Action Recognition,"Given the laparoscopic cholecystectomy image, ...","retract, dissect","retract, dissect, null verb, null target, null..."
4,/content/CholecT50/CholecT50/videos/VID36/0012...,Action Recognition,"Given the laparoscopic cholecystectomy image, ...",dissect,"dissect, coagulate, retract, dissect, coagulat..."
